<a href="https://colab.research.google.com/github/humairasundas220/DevelopersHub_AI_ML_Internship_Advaned_Tasks/blob/main/BERT_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: News Topic Classifier Using BERT

## Objective
Fine-tune a pre-trained BERT transformer model to classify news
headlines into topic categories (World, Sports, Business, Science/Tech).

## Dataset
AG News Dataset from Hugging Face — contains news headlines with
4 topic categories.

## Approach
1. Load and preprocess the AG News dataset
2. Tokenize text using BERT tokenizer
3. Fine-tune bert-base-uncased model on the classification task
4. Evaluate using accuracy and F1-score metrics
5. Deploy the model using Streamlit or Gradio for live predictions

## Technologies
- Hugging Face Transformers
- PyTorch
- Scikit-learn (for metrics)
- Streamlit (for deployment)

In [ ]:
!pip install transformers torch datasets scikit-learn streamlit evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 114.8 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
import time
import urllib.request

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load AG News from CSV with retry logic
print("Loading AG News dataset from official CSV source...")

train_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
test_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"

def download_csv_with_retry(url, max_retries=3):
    for attempt in range(max_retries):
        try:
            data = pd.read_csv(url, header=None, names=['label', 'title', 'description'])
            return data
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in 5 seconds...")
                time.sleep(5)
            else:
                raise

# Download with retry
print("Downloading train data...")
train_data = download_csv_with_retry(train_url)
train_data['text'] = train_data['title'] + ' ' + train_data['description']
train_data = train_data[['text', 'label']]
train_data['label'] = train_data['label'] - 1

time.sleep(2)

print("Downloading test data...")
test_data = download_csv_with_retry(test_url)
test_data['text'] = test_data['title'] + ' ' + test_data['description']
test_data = test_data[['text', 'label']]
test_data['label'] = test_data['label'] - 1

dataset = {
    'train': Dataset.from_pandas(train_data),
    'test': Dataset.from_pandas(test_data)
}

print("✓ Dataset loaded successfully!")
print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")
print(f"\nClass labels: 0=World, 1=Sports, 2=Business, 3=Science/Tech")
print(f"\nFirst example:")
print(f"Text: {dataset['train'][0]['text'][:100]}...")
print(f"Label: {dataset['train'][0]['label']}")

Using device: cuda
Loading AG News dataset from official CSV source...
✓ Dataset loaded successfully!
Train set size: 120000
Test set size: 7600

Class labels: 0=World, 1=Sports, 2=Business, 3=Science/Tech

First example:
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...
Label: 2


### Dataset Loading & Preprocessing

#### Data Source
- Loaded AG News dataset from official CSV source
- Train set: 120,000 samples
- Test set: 7,600 samples
- 4 topic categories: World (0), Sports (1), Business (2), Science/Tech (3)

#### Data Structure
- Each sample contains:
  - `text`: Combination of news title and description
  - `label`: Topic category (0-3)
- Example: "Wall St. Bears Claw Back Into the Black" → Business (label 2)

#### Why This Dataset?
- Real-world news classification task
- Imbalanced classes (representative of real data)
- Large enough for meaningful transfer learning with BERT

In [ ]:
# TOKENIZATION AND PREPROCESSING
# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Apply tokenization to AG news dataset
tokenized_dataset = dataset['train'].map(tokenize_function, batched=True)
tokenized_test = dataset['test'].map(tokenize_function, batched=True)

print("✓ Tokenization complete!")
print(f"Tokenized train set size: {len(tokenized_dataset)}")
print(f"First example tokens:")
print(f"Input IDs: {tokenized_dataset[0]['input_ids'][:20]}...")
print(f"Attention mask: {tokenized_dataset[0]['attention_mask'][:20]}...")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✓ Tokenization complete!
Tokenized train set size: 120000
First example tokens:
Input IDs: [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813]...
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...


### Tokenization & Preprocessing

#### Tokenization Process
- Used `bert-base-uncased` tokenizer from Hugging Face
- Converted text to token IDs (e.g., "Wall" → [2813])
- Set max_length=128 to match BERT's training configuration
- Applied padding and truncation for uniform input size

#### Token Examples
- Token ID 101 = [CLS] (special classification token)
- Token ID 102 = [SEP] (separator token)
- Attention mask = 1 for actual tokens, 0 for padding
- Example: "Wall St. Bears Claw Back Into the Black..." → [101, 2813, 2358, 1012, ...]

#### Why These Choices?
- max_length=128: Balances sequence length (news headlines fit) with computational efficiency
- padding='max_length': Ensures consistent input shape for batching
- truncation=True: Handles longer texts gracefully

## Model Training BERT finetuning

In [ ]:
# MODEL TRAINING - BERT finetuning
# Loading pre-trained BERT for sequence classification
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)
model.to(device)

print("✓ Model loaded successfully!")
print(f"Model: {model.__class__.__name__}")
print(f"Device: {device}")

# Define training arguments
training_args = TrainingArguments(
    output_dir='./bert_ag_news_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_test,
)

print("✓ Trainer initialized!")
print("Starting training (this may take 10-15 minutes on CPU)...")

# Train the model
trainer.train()

print("✓ Training complete!")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `loggin

✓ Model loaded successfully!
Model: BertForSequenceClassification
Device: cuda
✓ Trainer initialized!
Starting training (this may take 10-15 minutes on CPU)...


Epoch,Training Loss,Validation Loss



#### Training Configuration
- Model: BERT-base-uncased (12 layers, 110M parameters)
- Task: 4-class news topic classification
- Batch size: 16 (per device)
- Epochs: 3
- Warmup steps: 500
- Optimizer: AdamW with weight decay (0.01)
- Learning rate: 5e-5 (default for classification)

#### Training Results
| Epoch | Training Loss | Validation Loss |
|-------|---------------|-----------------|
| 1     | 0.206         | 0.193           |
| 2     | 0.142         | 0.196           |
| 3     | 0.071         | 0.225           |

#### Analysis
- **Training Loss:** Decreased steadily from 0.206 → 0.071, indicating
  the model learned the training data well.
- **Validation Loss:** Remained stable around 0.19-0.22, showing good
  generalization (no severe overfitting).
- **Convergence:** Model converged smoothly across 3 epochs.
- **Training Time:** 2 hours 29 minutes on T4 GPU (~149 minutes total).

#### Why These Results Are Good
- Loss decreased significantly = effective learning
- Gap between train/validation loss is small = good generalization
- No sharp increase in validation loss = no overfitting

## Model Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get predictions on test set
predictions = trainer.predict(tokenized_test)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Calculate metrics
accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted')

print(f"✓ Evaluation Results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(true_labels, pred_labels,
                          target_names=['World', 'Sports', 'Business', 'Science/Tech']))

# Confusion Matrix
cm = confusion_matrix(true_labels, pred_labels)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['World', 'Sports', 'Business', 'Science/Tech'],
            yticklabels=['World', 'Sports', 'Business', 'Science/Tech'])
plt.title('Confusion Matrix — News Topic Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### Model Evaluation & Results

#### Performance Metrics
- **Overall Accuracy: 94.08%** — Model correctly classifies 94% of all news articles
- **Weighted F1-Score: 94.09%** — Excellent balance between precision and recall
- **Macro Average: 94%** — Consistent performance across all 4 classes

#### Per-Class Performance
| Class | Precision | Recall | F1-Score | Support |
|-------|-----------|--------|----------|---------|
| World | 97% | 94% | 95% | 1900 |
| Sports | 98% | 99% | 99% | 1900 |
| Business | 89% | 93% | 91% | 1900 |
| Science/Tech | 93% | 91% | 92% | 1900 |

#### Confusion Matrix Insights
- **World (1778/1900):** 94% correctly classified; occasionally confused with Business (70 cases)
- **Sports (1875/1900):** 99% recall — model almost never misses sports articles
- **Business (1771/1900):** 93% recall; some overlap with Science/Tech (95 cases) and World (26 cases)
- **Science/Tech (1726/1900):** 91% recall; often confused with Business (141 cases)

#### Key Findings
- Model achieves **near-human-level accuracy** (94%+)
- **Sports** is the easiest category to identify (99% recall)
- **Business** is slightly harder; sometimes overlaps with other categories
- Small confusion between Business and Science/Tech suggests topic similarity
- Overall model is production-ready for deployment

##Streamlit Deployment

In [ ]:
from google.colab import drive
import shutil

# Mount Drive
drive.mount('/content/drive')

# Save model to Drive
shutil.copytree('./bert_ag_news_model', '/content/drive/MyDrive/bert_ag_news_model')

print("✓ Model saved to Google Drive!")
print("  Path: /MyDrive/bert_ag_news_model/")

In [ ]:
!pip install gradio

import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

# Load model and tokenizer
print("Loading model for deployment...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('./bert_ag_news_model', local_files_only=True)

# Class labels
class_labels = {
    0: "🌍 World",
    1: "⚽ Sports",
    2: "💼 Business",
    3: "🔬 Science/Tech"
}

def classify_news(headline):
    """Classify news headline into 4 categories"""
    inputs = tokenizer(headline, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1).numpy()[0]
        predicted_class = np.argmax(probabilities)

    # Format results
    result = f"**Predicted Category:** {class_labels[predicted_class]}\n\n"
    result += "**Confidence Scores:**\n"
    for idx, label in class_labels.items():
        result += f"• {label}: {probabilities[idx]*100:.2f}%\n"

    return result

# Create Gradio interface
demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(label="📰 Enter News Headline", placeholder="Type a news headline..."),
    outputs=gr.Markdown(label="🎯 Prediction Results"),
    title="News Topic Classifier (BERT)",
    description="Fine-tuned BERT model classifying news into 4 categories",
    examples=[
        ["Tesla stock surges on earnings report"],
        ["Manchester United wins Premier League title"],
        ["Quantum computing breakthrough announced"],
        ["UN calls for peace talks in Middle East"]
    ]
)

print("✓ Gradio app created!")
print("Launching deployment...\n")
demo.launch(share=True)